In [36]:
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

In [37]:
import pandas as pd

# Читаем файл построчно
with open('rental_18.csv', 'r', encoding='utf-8') as f:
    lines = [line.strip() for line in f if line.strip()]

rows = []
for line in lines[1:]:  # Пропускаем первую строку с заголовком
    # Убираем лишние внешние кавычки и лишние запятые в конце строки
    clean_line = line.rstrip(',').strip('"')
    # Заменяем двойные кавычки (экранированные Excel) на одинарные
    clean_line = clean_line.replace('""', '"')
    # Разделяем по первой точке с запятой на amount и text
    parts = clean_line.split(';', 1)
    if len(parts) == 2:
        rows.append([parts[0], parts[1].strip('"')])

df = pd.DataFrame(rows, columns=['amount', 'text'])
df.head()


,amount,text
0,4,Снимем жильё с 30 июня по 7 июля 3 взрослых и ...
1,5,"Ищем жилье! 2взрослых и 3 ребенка ( 6,6 и 4 го..."
2,6,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...
3,2,"Ищем жилье в самом Лазаревском для двоих,2-мес..."
4,3,"Ищем жильё для 2-х взрослых"",ребёнок (7 лет) и..."


In [38]:
results = []

for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain.invoke({"text": text})
        results.append(result)
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")

In [39]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

giga_key = os.getenv("GIGA_KEY")
print(f"GIGA_KEY: {giga_key}")


GIGA_KEY: OWU5OTAwNDYtZmU5Ny00YzU3LWJiNWQtMWRkMjliM2Q5YmExOmVjZTYxYzZjLWQ1YmYtNDllYi04YmZhLTQ3ZWY0NGQxOTQ2Ng==


In [40]:
llm = GigaChat(
credentials=giga_key,
model="GigaChat-2",
verify_ssl_certs=False,
temperature=0.2,
max_tokens=1000
)

In [41]:
# Проверка подключения
response = llm.invoke("Привет! Как дела?")
print(response.content)

Привет! Всё отлично, спасибо. Готов обсудить что-нибудь интересное или помочь разобраться с задачей. Чем займёмся?


In [42]:
from langchain_core.prompts import (
ChatPromptTemplate,
SystemMessagePromptTemplate,
HumanMessagePromptTemplate
)
# Шаблон системного сообщения
system_prompt = SystemMessagePromptTemplate.from_template(
"Ты — эксперт по анализу текстовых заявок на аренду жилья."
)
# Шаблон пользовательского сообщения
user_prompt = HumanMessagePromptTemplate.from_template(
"Заявка: {text}\n\nИзвлеки количество человек и верни только число."
)
# Собираем чат-шаблон
chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

In [43]:
# При вызове цепочки:
chain = chat_prompt | llm | StrOutputParser()
result = chain.invoke({"text": "Семья из трех человек снимает квартиру"})
print(result) # ожидаем "3"

3


In [44]:
# Простой промпт для извлечения количества людей
basic_prompt = PromptTemplate(
 input_variables=["text"],
 template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:"""
)
# Создание цепочки
chain = basic_prompt | llm | StrOutputParser()
# Тестирование
test_texts = [
 "Снимем жильё с 30 июня по 7 июля 3 взрослых и ребёнок",
    "Ищем жилье! 2взрослых и 3 ребенка ( 6, 6 и 4 года) можно 2 двухспалки, недалеко от центра и моря, эконом! С 11.07 по 23.07.",
    "Здравствуйте! Ищем жильё для 2 взрослых и 4 детей (8, 5, 4, 2) в период с 29.05.2018 по 09.06.2018 года с кухней, недалёко от моря! Предложения в л/с",
    "Ищем жилье в самом Лазаревском для двоих, 2-местный номер с удобствами, с 7-8 сентября не менее чем на неделю, 700 р. за номер",
    "Ищем жильё для 2-х взрослых, ребёнок (7 лет) и собака. Только Чёрное море, не Крым и Анапа.\nС 15 июля на 10 дней.\nПредложения в личку.",
    "Ищу жилье, интересует Пицунда, песчаный пляж, недалеко от моря. С 20-22 на 10 дней, 2 взрослых + 1 ребенок 2,5 года Рассмотрю все варианты.",
    "Лазаревское, Солоники - бюджетное жилье на 3 суток 4 взрослых с 5.09 интересует. Только в личку.",
    "Ищем жильё с 18-19 августа на 7 дней.Частный сектор.Обязательно с местом для парковки легковой машины.Нас 3 взрослых.",
    "Здравствуйте! Ищем жильё на августа числа с 6-7 на недельку. 2 взрослых и 2 ребенка 6 и 9 лет! Желательно номер в пределах 2000 или 500р с человека",
    "Добрый день!Ищем жилье с 25.04-05.05 2 взрослых + ребенок 3х лет!Душ и туалет в номере!",
    "здравствуйте. с 24 июля по 1 августа ищем жилье для 2-х взрослых эконом. стоимость подскажите, пожалуйста?",
    "Здравствуйте! Ищем жилье в частном секторе эконом на 2х человек в период с 4 по 13июля.(ул.Переулок Победы, ул.Свирская, ул.Седова). Рассмотрим все варианты.",
    "Ищем жилье, 4 человек, 3 взрослых и 1 ребенок 7 лет, с удобствами, двухкомнатный, либо два, двух местных, с удобствами и басейном, с 16 июля до 27 июля.",
    "Здравствуйте. Ищем жильё в период с 26 мая по 14 июня. Рядом с санаторием Бирюза. На одного человека и две маленькие собачки.",
    "Добрый день! Ищем жилье в Лазаревском с 02.07-10.07. Двухместный номер с удобствами. Вблизи моря, не окраина. До 2т.р./сут."
]
for text in test_texts:
 result = chain.invoke({"text": text})
 print(f"Текст: {text}")
 print(f"Результат: {result}")
 print("---")

Текст: Снимем жильё с 30 июня по 7 июля 3 взрослых и ребёнок
Результат: 4
---
Текст: Ищем жилье! 2взрослых и 3 ребенка ( 6, 6 и 4 года) можно 2 двухспалки, недалеко от центра и моря, эконом! С 11.07 по 23.07.
Результат: 5
---
Текст: Здравствуйте! Ищем жильё для 2 взрослых и 4 детей (8, 5, 4, 2) в период с 29.05.2018 по 09.06.2018 года с кухней, недалёко от моря! Предложения в л/с
Результат: 6
---
Текст: Ищем жилье в самом Лазаревском для двоих, 2-местный номер с удобствами, с 7-8 сентября не менее чем на неделю, 700 р. за номер
Результат: 2
---
Текст: Ищем жильё для 2-х взрослых, ребёнок (7 лет) и собака. Только Чёрное море, не Крым и Анапа.
С 15 июля на 10 дней.
Предложения в личку.
Результат: 4
---
Текст: Ищу жилье, интересует Пицунда, песчаный пляж, недалеко от моря. С 20-22 на 10 дней, 2 взрослых + 1 ребенок 2,5 года Рассмотрю все варианты.
Результат: 3
---
Текст: Лазаревское, Солоники - бюджетное жилье на 3 суток 4 взрослых с 5.09 интересует. Только в личку.
Результат: 4
---
Текст

In [ ]:
import pandas as pd

print(df.dtypes)

amount_num = pd.to_numeric(df["amount"], errors="coerce")
result_num = pd.to_numeric(df["result"], errors="coerce")

matches = amount_num == result_num
correct = int(matches.sum())
total = len(df)
errors = total - correct
accuracy = correct / total if total > 0 else 0

print(f"Верных ответов: {correct} из {total}")
print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")

amount    str
text      str
result    str
dtype: object
Верных ответов: 15 из 15
Ошибок: 0
Точность: 100.0%


In [56]:
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

json_schema = {
    "type": "object",
    "properties": {
        "count_adults": {"type": "integer", "description": "Число взрослых (0, если не указано)"},
        "count_children": {"type": "integer", "description": "Число детей (0, если не указано)"},
        "start_date": {
            "type": ["string", "null"],
            "description": "Дата заезда как в тексте (при диапазоне — только самая ранняя дата)"
        },
        "nights": {
            "type": ["integer", "null"],
            "description": "Число ночей проживания (или максимум при диапазоне)"
        },
        "price_per_day": {
            "type": ["number", "null"],
            "description": "Желаемая цена в сутки (при диапазоне или выборе — максимум)"
        },
        "remarks": {
            "type": ["string", "null"],
            "description": "Особые пожелания (животные, кроватка, удобства, локация и т.д.)"
        },
    },
    "required": [
        "count_adults",
        "count_children",
        "start_date",
        "nights",
        "price_per_day",
        "remarks",
    ],
}

json_parser = JsonOutputParser(schema=json_schema)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Ты — эксперт по анализу заявок на аренду жилья.
Извлекай данные строго по правилам:
1. count_adults и count_children: ВСЕГДА включай ОБА ключа в JSON (целые числа). Если дети не упомянуты — ставь 0. Если взрослые не упомянуты — ставь 0. "Пара" / "двое" без детей -> count_adults=2, count_children=0. "Один человек" -> count_adults=1, count_children=0.
2. start_date: бери дату заезда ТОЛЬКО из текста заявки, НЕ выдумывай год и НЕ подставляй месяц, если его нет в тексте. При диапазоне (например, "с 7-8 сентября", "с 20-22 числа") верни ТОЛЬКО первую (самую раннюю) дату начала ("7 сентября", "20"). Если даты нет — null.
3. nights: число ночей проживания (целое число).
   - Если указан период дат "с D1.M1 по D2.M2" (или "до D2.M2"):
     * в одном месяце: D2 - D1 (например, с 16 июля до 27 июля -> 11 ночей);
     * при переходе в следующий месяц: (дней_в_M1 - D1) + D2, учитывая число дней в месяце (апрель=30, май=31, июнь=30, июль=31, август=31). Например: с 25.04 по 05.05 -> (30 - 25) + 5 = 10; с 24 июля по 1 августа -> (31 - 24) + 1 = 8; с 26 мая по 14 июня -> (31 - 26) + 14 = 19.
   - Если указано "на N ночей / дней / суток / недельку" — возьми N (или максимум при диапазоне).
   - Если не указано — null.
4. price_per_day: число (желаемая цена за сутки). При диапазоне (например, "до 1500-2500 р/сут", "до 2т.р./сут.") бери максимум (2500, 2000). Если указаны две суммы (например, "2000 или 500р с человека"), ВСЕГДА выбирай БОЛЬШЕЕ число (2000). Если цены нет — null.
5. remarks: строка с особыми пожеланиями (кроватка, парковка, вайфай, животные, расположение у моря и т.д.). Если пожеланий нет — null."""
    ),
    (
        "human",
        """Заявка:
{text}

Верни строго JSON по схеме:
{format_instructions}"""
    ),
]).partial(format_instructions=json_parser.get_format_instructions())

chain = prompt | llm | json_parser

parsed_results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        res = chain.invoke({"text": text})
        res.setdefault("count_adults", 0)
        res.setdefault("count_children", 0)
        parsed_results.append(res)
    except Exception as e:
        parsed_results.append({"error": str(e)})

fields_df = pd.DataFrame(parsed_results)
df_full = pd.concat([df.reset_index(drop=True), fields_df], axis=1)

df_full["count_adults"] = pd.to_numeric(df_full["count_adults"], errors="coerce").fillna(0).astype(int)
df_full["count_children"] = pd.to_numeric(df_full["count_children"], errors="coerce").fillna(0).astype(int)

df_full.to_csv("rental_18_extracted.csv", index=False, encoding="utf-8-sig")
df_full

,amount,text,result,count_adults,count_children,start_date,nights,price_per_day,remarks
0,4,Снимем жильё с 30 июня по 7 июля 3 взрослых и ...,4,3,1,30 июня,7.0,NaN,NaN
1,5,"Ищем жилье! 2взрослых и 3 ребенка ( 6,6 и 4 го...",5,2,3,11.07,12.0,NaN,"можно 2 двухспалки, недалеко от центра и моря,..."
2,6,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...,6,2,4,29.05.2018,11.0,NaN,"жильё недалеко от моря, с кухней"
3,2,"Ищем жилье в самом Лазаревском для двоих,2-мес...",2,2,0,7 сентября,7.0,700.0,NaN
4,3,"Ищем жильё для 2-х взрослых"",ребёнок (7 лет) и...",3,2,1,NaN,NaN,NaN,"Чёрное море, проживание для семьи с ребёнком 7..."
5,3,"Ищу жилье,интересует Пицунда,песчаный пляж,нед...",3,2,1,20,10.0,NaN,"Пицунда, песчаный пляж, недалеко от моря"
6,4,"Лазаревское,Солоники - бюджетное жилье на 3 су...",4,4,0,5 сентября,3.0,NaN,NaN
7,3,Ищем жильё с 18-19 августа на 7 дней.Частный с...,3,3,0,18 августа,7.0,NaN,"Частный сектор, место для парковки легковой ма..."
8,4,Здравствуйте! Ищем жильё на августа числа с 6-...,4,2,2,6 августа,7.0,2000.0,номер в пределах 2000 или 500р с человека
9,3,Добрый день!Ищем жилье с 25.04-05.05 2 взрослы...,3,2,1,25 апреля,10.0,NaN,Душ и туалет в номере


In [54]:
import json

test_cases = [
    {
        "text": "Семья из пяти человек ищет большую квартиру с 10 по 17 июля, до 3000 руб в сутки.",
        "expected": {"count_adults": 5, "count_children": 0, "nights": 7, "price_per_day": 3000},
    },
    {
        "text": "Молодая пара с ребёнком 4 лет рассматривает жильё с 7-8 сентября на неделю, бюджет до 1500-2500 р/сут, нужна детская кроватка.",
        "expected": {"count_adults": 2, "count_children": 1, "start_date": "7 сентября", "nights": 7, "price_per_day": 2500},
    },
    {
        "text": "Нас трое взрослых, с 20-22 числа на 5 дней, обязательно парковка и вайфай.",
        "expected": {"count_adults": 3, "count_children": 0, "start_date": "20", "nights": 5, "price_per_day": None},
    },
    {
        "text": "Ищем студию для двоих с 1 по 10 августа, строго без животных.",
        "expected": {"count_adults": 2, "count_children": 0, "nights": 9, "price_per_day": None},
    },
    {
        "text": "Один человек, заезд 15.06 на 14 ночей, рядом с морем.",
        "expected": {"count_adults": 1, "count_children": 0, "start_date": "15.06", "nights": 14, "price_per_day": None},
    },
]

expected_keys = {
    "count_adults",
    "count_children",
    "start_date",
    "nights",
    "price_per_day",
    "remarks",
}

passed = 0
for idx, case in enumerate(test_cases, 1):
    text = case["text"]
    exp = case["expected"]
    try:
        res = chain.invoke({"text": text})
        res.setdefault("count_adults", 0)
        res.setdefault("count_children", 0)

        missing = expected_keys - set(res.keys())
        mismatches = []
        for k, v in exp.items():
            if res.get(k) != v:
                mismatches.append(f"{k}: получено {res.get(k)!r}, ожидалось {v!r}")

        if not missing and not mismatches:
            passed += 1
            status = "✅ ПРОЙДЕН"
        else:
            status = "❌ ЕСТЬ РАСХОЖДЕНИЯ"

        print(f"Тест #{idx} [{status}]: {json.dumps(res, ensure_ascii=False)}")
        if missing:
            print(f"   Не хватает ключей: {missing}")
        for m in mismatches:
            print(f"   Расхождение -> {m}")
    except Exception as e:
        print(f"Тест #{idx} [❌ ОШИБКА]: {e}")

print(f"\nИтог: пройдено {passed} из {len(test_cases)} тестов.")


Тест #1 [✅ ПРОЙДЕН]: {"count_adults": 5, "count_children": 0, "start_date": "10 июля", "nights": 7, "price_per_day": 3000, "remarks": null}
Тест #2 [✅ ПРОЙДЕН]: {"count_adults": 2, "count_children": 1, "start_date": "7 сентября", "nights": 7, "price_per_day": 2500, "remarks": "детская кроватка"}
Тест #3 [✅ ПРОЙДЕН]: {"count_adults": 3, "count_children": 0, "start_date": "20", "nights": 5, "price_per_day": null, "remarks": "обязательно парковка и вайфай"}
Тест #4 [✅ ПРОЙДЕН]: {"count_adults": 2, "count_children": 0, "start_date": "1 августа", "nights": 9, "price_per_day": null, "remarks": "строго без животных"}
Тест #5 [✅ ПРОЙДЕН]: {"count_adults": 1, "count_children": 0, "start_date": "15.06", "nights": 14, "price_per_day": null, "remarks": "рядом с морем"}

Итог: пройдено 5 из 5 тестов.
